# CaPMeS: Calibrated Prototype Meta-Selector for Long-Term Forecasting

This notebook implements a **new, research-grounded direction** after these failed ideas:

- PBT search
- DBLoss full run
- adaptive lookback full run
- gated residual pilot
- HeRSMix pilot
- PCWRA pilot

## Why this direction

Your results already show that **different models win on different datasets**, which matches recent evidence that there are **no universal champions** in LTSF. The more plausible next step is **model selection / model combination under regime features**, not another end-to-end monolithic backbone.

This notebook uses:

1. **Independent experts** trained separately:
   - `seasonal_naive`
   - `dlinear_target`
   - `local_conv`
   - `wavelet_expert` (your best wavelet architecture)

2. **Validation meta-dataset**:
   - collect each expert's validation errors
   - define hard label = best expert per validation window
   - define soft target = softmax over negative validation errors

3. **Prototype-conditioned selector**:
   - fit KMeans prototypes on training-window features
   - selector uses window features + distances to prototypes
   - train with:
     - hard CE to best expert
     - soft distillation to expert-error soft targets
     - selective weighting using expert-margin confidence

4. **Final prediction**
   - either hard model selection (default)
   - or soft combination (optional)

## Strongest defensible claim
A **calibrated prototype meta-selector** can outperform a fixed expert by selecting the most suitable forecasting inductive bias for each window.

## Weakest claim you should NOT make
Do **not** claim a universal new forecasting backbone. This is a **model-selection / meta-forecasting** method.

## Kill test
If the selector does **not** beat:
- the strongest internal expert on at least 3/4 pilot datasets, and
- the simple DLinear expert on at least 3/4 pilot datasets,

reject the idea.

In [ ]:
# ============================================
# CONFIG
# ============================================

RUN_MODE = "pilot_then_full"   # options: "pilot_only", "pilot_then_full", "full_only"
AUTO_FULL_IF_PILOT_GOOD = True

PILOT_DATASETS = ["etth1", "weather", "traffic", "ili"]

# pilot success rule:
# 1) selector beats strongest internal expert on >= 3 pilot datasets
# 2) selector beats dlinear_target on >= 3 pilot datasets
PILOT_THRESH_BEST_INTERNAL = 3
PILOT_THRESH_DLINEAR = 3

FINAL_PRED_MODE = "hard"   # options: "hard", "soft"

SEED = 42
ROOT_DIR = "/data/Sajjan_Singh/spml/wavelet_seq_project"

In [ ]:
# ============================================
# IMPORTS AND PROJECT SETUP
# ============================================

from pathlib import Path
import sys
import importlib
import gc
import json
import random
import copy
import time
import math

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.cluster import KMeans

ROOT = Path(ROOT_DIR).resolve()
SCRIPTS_DIR = ROOT / "scripts"
RUNNER_PATH = SCRIPTS_DIR / "phase5_adaptive_wavelet_model.py"

if not RUNNER_PATH.exists():
    raise FileNotFoundError(f"Missing runner file: {RUNNER_PATH}")

if str(SCRIPTS_DIR) not in sys.path:
    sys.path.append(str(SCRIPTS_DIR))

import phase5_adaptive_wavelet_model as p5
importlib.reload(p5)

OUT_TABLES = ROOT / "results" / "tables" / "phase13_caps"
OUT_PREDS = ROOT / "results" / "predictions" / "phase13_caps"
OUT_CKPTS = ROOT / "results" / "checkpoints" / "phase13_caps"

OUT_TABLES.mkdir(parents=True, exist_ok=True)
OUT_PREDS.mkdir(parents=True, exist_ok=True)
OUT_CKPTS.mkdir(parents=True, exist_ok=True)

BEST_PATH = ROOT / "results" / "tables" / "master_best_by_dataset_unified.csv"

if torch.cuda.is_available():
    torch.backends.cudnn.benchmark = True
    if hasattr(torch, "set_float32_matmul_precision"):
        torch.set_float32_matmul_precision("high")
    GPU_NAME = torch.cuda.get_device_name(0)
    GPU_MEM_GB = torch.cuda.get_device_properties(0).total_memory / (1024**3)
else:
    GPU_NAME = "CPU"
    GPU_MEM_GB = 0.0

USE_AMP = bool(torch.cuda.is_available()) and bool(getattr(p5, "TRAIN_CFG", {}).get("use_amp", True))
PATIENCE = int(getattr(p5, "TRAIN_CFG", {}).get("full_patience", 8))

print("Imported p5 from:", RUNNER_PATH)
print("DEVICE:", p5.DEVICE)
print("GPU_NAME:", GPU_NAME)
print("GPU_MEM_GB:", round(GPU_MEM_GB, 2))
print("USE_AMP:", USE_AMP)
print("PATIENCE:", PATIENCE)
print("All datasets:", p5.ALL_DATASETS)

In [ ]:
# ============================================
# FIXED SETTINGS
# ============================================

ALL_DATASETS = list(p5.ALL_DATASETS)

# best wavelet architecture from your ablations
WAVE_ARCH = {
    "levels": 2,
    "wavelet_family": "Db4",
    "d_model": 128,
    "num_backbone_blocks": 2,
    "dropout": 0.10,
    "filter_reg_lambda": 1e-4,
    "gate_entropy_lambda": 1e-4,
}

if GPU_MEM_GB >= 80:
    BATCH_MUL = {
        "etth1": 2, "etth2": 2, "ettm1": 2, "ettm2": 2,
        "weather": 2, "electricity": 1, "traffic": 1,
        "exchange": 2, "ili": 2,
    }
else:
    BATCH_MUL = {ds: 1 for ds in ALL_DATASETS}

SELECTOR_CFG = {
    "num_prototypes": 4,
    "hidden": 64,
    "temp_soft_targets": 0.50,
    "temp_soft_pred": 0.50,
    "lambda_soft": 0.50,
    "lambda_entropy": 1e-4,
    "keep_ratio": 0.75,
}

print("WAVE_ARCH:", WAVE_ARCH)
print("SELECTOR_CFG:", SELECTOR_CFG)
print("BATCH_MUL:", BATCH_MUL)

In [ ]:
# ============================================
# HELPERS
# ============================================

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

def mse(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=np.float64)
    y_pred = np.asarray(y_pred, dtype=np.float64)
    return float(np.mean((y_true - y_pred) ** 2))

def mae(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=np.float64)
    y_pred = np.asarray(y_pred, dtype=np.float64)
    return float(np.mean(np.abs(y_true - y_pred)))

def rmse(y_true, y_pred):
    return float(np.sqrt(mse(y_true, y_pred)))

def resolve_seq_len(ds):
    seq_candidates = p5.resolve_seq_candidates(ds, "searched")
    return int(seq_candidates[min(1, len(seq_candidates) - 1)])

def resolve_batch_size(ds):
    return int(p5.DEFAULT_BATCH_SIZE[ds] * BATCH_MUL[ds])

def resolve_epochs(ds):
    return int(p5.DEFAULT_EPOCHS[ds])

def resolve_train_end_from_bundle_or_disk(dataset_name, bundle):
    if isinstance(bundle, dict):
        for k in ["train_end", "train_rows", "n_train", "train_size"]:
            if k in bundle:
                return int(bundle[k])

        if "split_indices" in bundle and isinstance(bundle["split_indices"], dict):
            si = bundle["split_indices"]
            for k in ["train_end", "train_rows", "n_train", "train_size"]:
                if k in si:
                    return int(si[k])

    split_json = ROOT / "data" / "splits" / f"{dataset_name}_splits.json"
    if split_json.exists():
        data = json.loads(split_json.read_text())
        for k in ["train_end", "train_rows", "n_train", "train_size"]:
            if k in data:
                return int(data[k])

    fallback_train_rows = {
        "etth1": 8640, "etth2": 8640,
        "ettm1": 34560, "ettm2": 34560,
        "weather": 36886,
        "electricity": 18412,
        "traffic": 12280,
        "exchange": 5311,
        "ili": 676,
    }
    return int(fallback_train_rows[dataset_name])

def moving_avg_1d(x, kernel):
    if kernel <= 1:
        return x
    pad = kernel // 2
    x3 = x.unsqueeze(1)
    xpad = F.pad(x3, (pad, pad), mode="replicate")
    out = F.avg_pool1d(xpad, kernel_size=kernel, stride=1)
    return out.squeeze(1)

def patchify(x, patch_size):
    B, H = x.shape
    n = H // patch_size
    x = x[:, :n * patch_size]
    return x.reshape(B, n, patch_size)

def patchwise_structural_loss(pred, true, patch_size=12):
    pred_p = patchify(pred, patch_size)
    true_p = patchify(true, patch_size)

    mu_p = pred_p.mean(dim=-1)
    mu_t = true_p.mean(dim=-1)
    var_p = pred_p.var(dim=-1, unbiased=False)
    var_t = true_p.var(dim=-1, unbiased=False)

    p_center = pred_p - mu_p.unsqueeze(-1)
    t_center = true_p - mu_t.unsqueeze(-1)
    corr_num = (p_center * t_center).mean(dim=-1)
    corr_den = p_center.pow(2).mean(dim=-1).sqrt() * t_center.pow(2).mean(dim=-1).sqrt() + 1e-6
    corr = corr_num / corr_den

    return (
        F.mse_loss(mu_p, mu_t)
        + F.mse_loss(var_p, var_t)
        + ((1.0 - corr) ** 2).mean()
    )

def haar_multiscale_features(x):
    feats = [x]
    cur = x
    for _ in range(3):
        if cur.shape[1] < 2:
            break
        even = cur[:, 0::2]
        odd = cur[:, 1::2]
        m = min(even.shape[1], odd.shape[1])
        even = even[:, :m]
        odd = odd[:, :m]
        approx = (even + odd) / 2.0
        detail = (even - odd) / 2.0
        feats.append(approx)
        feats.append(detail)
        cur = approx

    out = []
    for f in feats:
        out.append(f.mean(dim=1, keepdim=True))
        out.append(f.std(dim=1, keepdim=True))
        out.append(f.abs().mean(dim=1, keepdim=True))
        out.append((f ** 2).mean(dim=1, keepdim=True))
    return torch.cat(out, dim=1)

def selector_features_torch(x_target):
    spec = torch.fft.rfft(x_target, dim=1)
    power = spec.real.pow(2) + spec.imag.pow(2)
    total_power = power.sum(dim=1) + 1e-8
    Fbins = power.shape[1]
    split = max(1, int(round(Fbins * 0.50)))
    high_ratio = power[:, split:].sum(dim=1) / total_power

    mean_abs = x_target.abs().mean(dim=1)
    std = x_target.std(dim=1)
    slope = x_target[:, -1] - x_target[:, 0] if x_target.shape[1] >= 2 else torch.zeros_like(mean_abs)

    if x_target.shape[1] >= 2:
        x1 = x_target[:, 1:]
        x0 = x_target[:, :-1]
        num = (x1 * x0).mean(dim=1)
        den = x1.pow(2).mean(dim=1).sqrt() * x0.pow(2).mean(dim=1).sqrt() + 1e-6
        lag1 = num / den
    else:
        lag1 = torch.zeros_like(mean_abs)

    wave = haar_multiscale_features(x_target)

    feats = torch.cat([
        mean_abs.unsqueeze(1),
        std.unsqueeze(1),
        slope.unsqueeze(1),
        high_ratio.unsqueeze(1),
        lag1.unsqueeze(1),
        wave
    ], dim=1)
    return feats

def selector_features_np(x_np):
    x = torch.tensor(x_np, dtype=torch.float32)
    return selector_features_torch(x).numpy().astype(np.float32)

def metric_row_from_npz(dataset, model, family, seq_len, pred_len, pred_file_rel):
    p = ROOT / str(pred_file_rel)
    arr = np.load(p, allow_pickle=True)
    preds = np.asarray(arr["preds"], dtype=np.float64).reshape(-1)
    trues = np.asarray(arr["trues"], dtype=np.float64).reshape(-1)
    return {
        "dataset": dataset,
        "model": model,
        "family": family,
        "seq_len": int(seq_len),
        "pred_len": int(pred_len),
        "test_mse": mse(trues, preds),
        "test_mae": mae(trues, preds),
        "test_rmse": rmse(trues, preds),
        "prediction_file": str(pred_file_rel),
    }

In [ ]:
# ============================================
# EXPERTS
# ============================================

class SeasonalNaiveExpert(nn.Module):
    def __init__(self, pred_len):
        super().__init__()
        self.pred_len = int(pred_len)

    def forward(self, x_target):
        B, T = x_target.shape
        if T >= self.pred_len:
            return x_target[:, -self.pred_len:]
        else:
            last = x_target[:, -1:].repeat(1, self.pred_len)
            return last

class DLinearTargetOnly(nn.Module):
    def __init__(self, seq_len, pred_len, kernel_size=25):
        super().__init__()
        self.seq_len = int(seq_len)
        self.pred_len = int(pred_len)
        self.kernel_size = int(kernel_size)
        self.trend_linear = nn.Linear(self.seq_len, self.pred_len)
        self.seasonal_linear = nn.Linear(self.seq_len, self.pred_len)

    def forward(self, x_target):
        trend = moving_avg_1d(x_target, self.kernel_size)
        seasonal = x_target - trend
        return self.trend_linear(trend) + self.seasonal_linear(seasonal)

class LocalConvExpert(nn.Module):
    def __init__(self, seq_len, pred_len, hidden=64, kernel=5):
        super().__init__()
        self.conv1 = nn.Conv1d(1, hidden, kernel_size=kernel, padding=kernel // 2)
        self.conv2 = nn.Conv1d(hidden, hidden, kernel_size=kernel, padding=kernel // 2)
        self.proj = nn.Linear(seq_len * hidden, pred_len)

    def forward(self, x_target):
        x = x_target.unsqueeze(1)
        x = F.gelu(self.conv1(x))
        x = F.gelu(self.conv2(x))
        x = x.flatten(1)
        return self.proj(x)

class WaveletExpert(nn.Module):
    def __init__(self, input_dim, pred_len):
        super().__init__()
        self.model = p5.AdaptiveWaveletMixer(
            input_dim=int(input_dim),
            pred_len=int(pred_len),
            d_model=int(WAVE_ARCH["d_model"]),
            levels=int(WAVE_ARCH["levels"]),
            wavelet_family=WAVE_ARCH["wavelet_family"],
            num_backbone_blocks=int(WAVE_ARCH["num_backbone_blocks"]),
            dropout=float(WAVE_ARCH["dropout"]),
            filter_reg_lambda=float(WAVE_ARCH["filter_reg_lambda"]),
            gate_entropy_lambda=float(WAVE_ARCH["gate_entropy_lambda"]),
        )

    def forward(self, x_scaled):
        pred_scaled, aux = self.model(x_scaled)
        return pred_scaled

class ExpertWrapper(nn.Module):
    def __init__(self, expert_name, seq_len, pred_len, input_dim, target_idx):
        super().__init__()
        self.expert_name = expert_name
        self.target_idx = int(target_idx)

        if expert_name == "seasonal_naive":
            self.model = SeasonalNaiveExpert(pred_len=pred_len)
        elif expert_name == "dlinear_target":
            self.model = DLinearTargetOnly(seq_len=seq_len, pred_len=pred_len, kernel_size=25)
        elif expert_name == "local_conv":
            self.model = LocalConvExpert(seq_len=seq_len, pred_len=pred_len, hidden=64, kernel=5)
        elif expert_name == "wavelet_expert":
            self.model = WaveletExpert(input_dim=input_dim, pred_len=pred_len)
        else:
            raise ValueError(f"Unknown expert_name: {expert_name}")

    def forward(self, x_scaled):
        if self.expert_name == "wavelet_expert":
            return self.model(x_scaled)
        else:
            x_target = x_scaled[:, :, self.target_idx]
            return self.model(x_target)

In [ ]:
# ============================================
# SELECTOR
# ============================================

class MetaSelector(nn.Module):
    def __init__(self, feature_dim, num_prototypes, num_experts, hidden=64):
        super().__init__()
        self.num_prototypes = int(num_prototypes)
        self.num_experts = int(num_experts)

        self.register_buffer("prototype_centers", torch.zeros(num_prototypes, feature_dim))

        self.net = nn.Sequential(
            nn.Linear(feature_dim + num_prototypes, hidden),
            nn.GELU(),
            nn.Linear(hidden, hidden),
            nn.GELU(),
            nn.Linear(hidden, num_experts),
        )

    def set_prototypes(self, centers_np):
        centers = torch.tensor(centers_np, dtype=torch.float32)
        if centers.shape[0] < self.num_prototypes:
            pad = centers[-1:, :].repeat(self.num_prototypes - centers.shape[0], 1)
            centers = torch.cat([centers, pad], dim=0)
        elif centers.shape[0] > self.num_prototypes:
            centers = centers[:self.num_prototypes]
        self.prototype_centers = centers

    def forward(self, feat):
        dist = torch.cdist(feat, self.prototype_centers.to(feat.device), p=2)
        x = torch.cat([feat, dist], dim=1)
        logits = self.net(x)
        return logits, dist

def fit_selector_prototypes(bundle, seq_len, dataset_name, num_prototypes=4, max_samples=4000, seed=42):
    values = bundle["values_scaled"]
    target_idx = int(bundle["target_idx"])
    train_end = resolve_train_end_from_bundle_or_disk(dataset_name, bundle)
    x = values[:train_end, target_idx]

    windows = []
    max_start = max(1, len(x) - seq_len)
    step = max(1, max_start // max_samples)

    for s in range(0, max_start, step):
        w = x[s:s+seq_len]
        if len(w) == seq_len:
            windows.append(w)
        if len(windows) >= max_samples:
            break

    if len(windows) == 0:
        raise RuntimeError(f"No windows found for selector prototypes: {dataset_name}")

    windows = np.stack(windows, axis=0).astype(np.float32)
    feats = selector_features_np(windows)

    k = min(num_prototypes, len(feats))
    if k < 2:
        k = 1

    km = KMeans(n_clusters=k, random_state=seed, n_init=10)
    km.fit(feats)
    centers = km.cluster_centers_.astype(np.float32)

    if k < num_prototypes:
        pad = np.repeat(centers[-1:, :], num_prototypes - k, axis=0)
        centers = np.concatenate([centers, pad], axis=0)

    return centers

In [ ]:
# ============================================
# TRAINING / EVALUATION UTILITIES
# ============================================

EXPERT_NAMES = ["seasonal_naive", "dlinear_target", "local_conv", "wavelet_expert"]

def expert_trainable(name):
    return name != "seasonal_naive"

def expert_loss(pred, y):
    return F.mse_loss(pred, y) + 0.10 * patchwise_structural_loss(pred, y, patch_size=12)

@torch.no_grad()
def eval_expert_model(model, expert_name, loader, target_mean, target_std):
    model.eval()
    preds = []
    trues = []
    per_sample_mse = []

    for batch in loader:
        x = batch["x_scaled"].to(p5.DEVICE, non_blocking=True)
        y_scaled = batch["y_scaled"].to(p5.DEVICE, non_blocking=True)
        y_raw = batch["y_raw"].cpu().numpy()

        with torch.cuda.amp.autocast(enabled=USE_AMP):
            pred_scaled = model(x)

        pred_raw = pred_scaled.detach().cpu().numpy() * target_std + target_mean
        preds.append(pred_raw)
        trues.append(y_raw)

        mse_s = ((pred_scaled - y_scaled) ** 2).mean(dim=1).detach().cpu().numpy()
        per_sample_mse.append(mse_s)

    preds = np.concatenate(preds, axis=0)
    trues = np.concatenate(trues, axis=0)
    per_sample_mse = np.concatenate(per_sample_mse, axis=0)

    return {
        "preds": preds,
        "trues": trues,
        "mse": mse(trues.reshape(-1), preds.reshape(-1)),
        "mae": mae(trues.reshape(-1), preds.reshape(-1)),
        "rmse": rmse(trues.reshape(-1), preds.reshape(-1)),
        "per_sample_mse_scaled": per_sample_mse,
    }

def train_one_expert(dataset_name, expert_name):
    set_seed(SEED)

    bundle = p5.load_bundle(dataset_name, input_mode="multivariate")
    pred_len = p5.resolve_pred_len(dataset_name, "long")
    seq_len = resolve_seq_len(dataset_name)
    batch_size = resolve_batch_size(dataset_name)
    epochs = resolve_epochs(dataset_name)

    train_loader, val_loader, test_loader = p5.make_loaders(bundle, seq_len, pred_len, batch_size)

    target_idx = int(bundle["target_idx"])
    target_mean = float(bundle["scaler_mean"][target_idx])
    target_std = float(bundle["scaler_std"][target_idx])

    model = ExpertWrapper(
        expert_name=expert_name,
        seq_len=seq_len,
        pred_len=pred_len,
        input_dim=int(bundle["values_scaled"].shape[1]),
        target_idx=target_idx,
    ).to(p5.DEVICE)

    if not expert_trainable(expert_name):
        val_eval = eval_expert_model(model, expert_name, val_loader, target_mean, target_std)
        test_eval = eval_expert_model(model, expert_name, test_loader, target_mean, target_std)
        return {
            "dataset": dataset_name,
            "expert": expert_name,
            "model": model,
            "seq_len": seq_len,
            "pred_len": pred_len,
            "bundle": bundle,
            "target_mean": target_mean,
            "target_std": target_std,
            "val_eval": val_eval,
            "test_eval": test_eval,
            "best_epoch": 0,
            "train_seconds": 0.0,
        }

    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-5, foreach=False)
    scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)

    best_val_rmse = float("inf")
    best_state = None
    best_epoch = -1
    patience_left = PATIENCE
    t0 = time.time()

    for epoch in range(1, epochs + 1):
        model.train()
        loss_vals = []

        for batch in train_loader:
            x = batch["x_scaled"].to(p5.DEVICE, non_blocking=True)
            y = batch["y_scaled"].to(p5.DEVICE, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)

            with torch.cuda.amp.autocast(enabled=USE_AMP):
                pred = model(x)
                loss = expert_loss(pred, y)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            loss_vals.append(float(loss.detach().cpu().item()))

        val_eval = eval_expert_model(model, expert_name, val_loader, target_mean, target_std)
        val_rmse = float(val_eval["rmse"])

        print(f"[{dataset_name} | {expert_name}] epoch {epoch:03d} | train_loss={np.mean(loss_vals):.6f} | val_rmse={val_rmse:.6f}")

        if val_rmse < best_val_rmse:
            best_val_rmse = val_rmse
            best_state = copy.deepcopy(model.state_dict())
            best_epoch = epoch
            patience_left = PATIENCE
        else:
            patience_left -= 1
            if patience_left <= 0:
                print(f"[{dataset_name} | {expert_name}] early stopping at epoch {epoch}")
                break

    model.load_state_dict(best_state)
    val_eval = eval_expert_model(model, expert_name, val_loader, target_mean, target_std)
    test_eval = eval_expert_model(model, expert_name, test_loader, target_mean, target_std)

    return {
        "dataset": dataset_name,
        "expert": expert_name,
        "model": model,
        "seq_len": seq_len,
        "pred_len": pred_len,
        "bundle": bundle,
        "target_mean": target_mean,
        "target_std": target_std,
        "val_eval": val_eval,
        "test_eval": test_eval,
        "best_epoch": best_epoch,
        "train_seconds": float(time.time() - t0),
    }

In [ ]:
# ============================================
# META-DATASET AND SELECTOR TRAINING
# ============================================

def build_selector_dataset(dataset_name, expert_runs):
    # assume all expert runs share same bundle / loaders settings
    bundle = expert_runs[0]["bundle"]
    seq_len = expert_runs[0]["seq_len"]
    pred_len = expert_runs[0]["pred_len"]

    batch_size = resolve_batch_size(dataset_name)
    train_loader, val_loader, test_loader = p5.make_loaders(bundle, seq_len, pred_len, batch_size)

    # validation feature matrix from input histories
    val_feats = []
    test_feats = []

    for batch in val_loader:
        x = batch["x_scaled"]
        x_target = x[:, :, int(bundle["target_idx"])]
        val_feats.append(selector_features_torch(x_target).numpy())

    for batch in test_loader:
        x = batch["x_scaled"]
        x_target = x[:, :, int(bundle["target_idx"])]
        test_feats.append(selector_features_torch(x_target).numpy())

    val_feats = np.concatenate(val_feats, axis=0).astype(np.float32)
    test_feats = np.concatenate(test_feats, axis=0).astype(np.float32)

    # collect expert val/test predictions and errors
    val_preds = []
    test_preds = []
    val_errors = []

    for r in expert_runs:
        val_preds.append(r["val_eval"]["preds"])
        test_preds.append(r["test_eval"]["preds"])
        val_errors.append(r["val_eval"]["per_sample_mse_scaled"])

    val_preds = np.stack(val_preds, axis=1)    # [N, E, H]
    test_preds = np.stack(test_preds, axis=1)  # [N, E, H]
    val_errors = np.stack(val_errors, axis=1)  # [N, E]

    # hard best expert
    y_hard = val_errors.argmin(axis=1).astype(np.int64)

    # soft targets from negative errors
    soft = np.exp(-val_errors / SELECTOR_CFG["temp_soft_targets"])
    y_soft = soft / (soft.sum(axis=1, keepdims=True) + 1e-8)

    # confidence margin
    sort_err = np.sort(val_errors, axis=1)
    margin = sort_err[:, 1] - sort_err[:, 0]
    thresh = np.quantile(margin, 1.0 - SELECTOR_CFG["keep_ratio"])
    keep = (margin >= thresh).astype(np.float32)

    return {
        "val_feats": val_feats,
        "test_feats": test_feats,
        "val_preds": val_preds,
        "test_preds": test_preds,
        "val_errors": val_errors,
        "y_hard": y_hard,
        "y_soft": y_soft.astype(np.float32),
        "keep": keep.astype(np.float32),
    }

def train_selector(dataset_name, selector_data, bundle, seq_len, pred_len):
    set_seed(SEED)

    centers = fit_selector_prototypes(
        bundle=bundle,
        seq_len=seq_len,
        dataset_name=dataset_name,
        num_prototypes=SELECTOR_CFG["num_prototypes"],
        max_samples=4000,
        seed=SEED,
    )

    feat_dim = selector_data["val_feats"].shape[1]
    num_experts = len(EXPERT_NAMES)

    selector = MetaSelector(
        feature_dim=feat_dim,
        num_prototypes=SELECTOR_CFG["num_prototypes"],
        num_experts=num_experts,
        hidden=SELECTOR_CFG["hidden"],
    ).to(p5.DEVICE)
    selector.set_prototypes(centers)

    optimizer = torch.optim.AdamW(selector.parameters(), lr=1e-3, weight_decay=1e-5, foreach=False)

    X = torch.tensor(selector_data["val_feats"], dtype=torch.float32, device=p5.DEVICE)
    y_hard = torch.tensor(selector_data["y_hard"], dtype=torch.long, device=p5.DEVICE)
    y_soft = torch.tensor(selector_data["y_soft"], dtype=torch.float32, device=p5.DEVICE)
    keep = torch.tensor(selector_data["keep"], dtype=torch.float32, device=p5.DEVICE)

    best_loss = float("inf")
    best_state = None
    best_epoch = -1
    patience_left = PATIENCE

    N = X.shape[0]
    batch = min(1024, N)

    for epoch in range(1, 201):
        selector.train()
        perm = torch.randperm(N, device=p5.DEVICE)
        losses = []

        for i in range(0, N, batch):
            idx = perm[i:i+batch]
            xb = X[idx]
            yb_h = y_hard[idx]
            yb_s = y_soft[idx]
            kb = keep[idx]

            optimizer.zero_grad(set_to_none=True)

            logits, dist = selector(xb)
            logp = F.log_softmax(logits, dim=1)
            prob = torch.softmax(logits / SELECTOR_CFG["temp_soft_pred"], dim=1)

            ce = F.cross_entropy(logits, yb_h, reduction="none")
            ce = (ce * kb).sum() / (kb.sum() + 1e-6)

            kl = F.kl_div(F.log_softmax(logits, dim=1), yb_s, reduction="batchmean")

            ent = -(prob.clamp_min(1e-8).log() * prob).sum(dim=1).mean()

            loss = ce + SELECTOR_CFG["lambda_soft"] * kl + SELECTOR_CFG["lambda_entropy"] * ent
            loss.backward()
            optimizer.step()
            losses.append(float(loss.detach().cpu().item()))

        mean_loss = float(np.mean(losses))

        if epoch % 10 == 0 or epoch == 1:
            print(f"[{dataset_name} | selector] epoch {epoch:03d} | loss={mean_loss:.6f}")

        if mean_loss < best_loss:
            best_loss = mean_loss
            best_state = copy.deepcopy(selector.state_dict())
            best_epoch = epoch
            patience_left = PATIENCE
        else:
            patience_left -= 1
            if patience_left <= 0:
                print(f"[{dataset_name} | selector] early stopping at epoch {epoch}")
                break

    selector.load_state_dict(best_state)
    return selector, centers, best_epoch

@torch.no_grad()
def eval_selector(selector, selector_data):
    selector.eval()

    X_test = torch.tensor(selector_data["test_feats"], dtype=torch.float32, device=p5.DEVICE)
    logits, dist = selector(X_test)
    probs = torch.softmax(logits / SELECTOR_CFG["temp_soft_pred"], dim=1).cpu().numpy()

    test_preds = selector_data["test_preds"]  # [N,E,H]

    if FINAL_PRED_MODE == "hard":
        idx = probs.argmax(axis=1)
        chosen = np.arange(len(idx))
        pred = test_preds[chosen, idx, :]
    else:
        pred = (probs[:, :, None] * test_preds).sum(axis=1)

    return {
        "pred": pred,
        "probs": probs,
        "avg_probs": probs.mean(axis=0),
    }

In [ ]:
# ============================================
# RUN ONE DATASET
# ============================================

def run_dataset(dataset_name, modes=("base_only", "selector")):
    # Train independent experts
    expert_runs = []
    for expert_name in EXPERT_NAMES:
        print("\n" + "=" * 120)
        print(f"TRAIN EXPERT | dataset={dataset_name} | expert={expert_name}")
        print("=" * 120)
        r = train_one_expert(dataset_name, expert_name)
        expert_runs.append(r)

    # Build baseline rows from experts
    rows = []
    for r in expert_runs:
        test_eval = r["test_eval"]
        rows.append({
            "dataset": dataset_name,
            "mode": r["expert"],
            "model": f"CaPMeS_{r['expert']}",
            "family": "meta_selection_expert_pool",
            "seq_len": r["seq_len"],
            "pred_len": r["pred_len"],
            "batch_size": resolve_batch_size(dataset_name),
            "best_epoch": r["best_epoch"],
            "test_mse": float(test_eval["mse"]),
            "test_mae": float(test_eval["mae"]),
            "test_rmse": float(test_eval["rmse"]),
            "avg_selection_probs_json": "",
            "prediction_file": "",
            "checkpoint": "",
        })

    # selector dataset
    selector_data = build_selector_dataset(dataset_name, expert_runs)
    bundle = expert_runs[0]["bundle"]
    seq_len = expert_runs[0]["seq_len"]
    pred_len = expert_runs[0]["pred_len"]
    target_mean = expert_runs[0]["target_mean"]
    target_std = expert_runs[0]["target_std"]

    # selector
    print("\n" + "=" * 120)
    print(f"TRAIN SELECTOR | dataset={dataset_name}")
    print("=" * 120)
    selector, centers, selector_best_epoch = train_selector(
        dataset_name=dataset_name,
        selector_data=selector_data,
        bundle=bundle,
        seq_len=seq_len,
        pred_len=pred_len,
    )

    sel_eval = eval_selector(selector, selector_data)

    # test truth from any expert run
    trues = expert_runs[0]["test_eval"]["trues"]
    pred = sel_eval["pred"]

    sel_mse = mse(trues.reshape(-1), pred.reshape(-1))
    sel_mae = mae(trues.reshape(-1), pred.reshape(-1))
    sel_rmse = rmse(trues.reshape(-1), pred.reshape(-1))

    # save selector artifacts
    ckpt_path = OUT_CKPTS / f"{dataset_name}_selector.pt"
    pred_path = OUT_PREDS / f"{dataset_name}_selector_test_predictions.npz"

    torch.save({
        "dataset": dataset_name,
        "selector_cfg": SELECTOR_CFG,
        "expert_names": EXPERT_NAMES,
        "prototype_centers": centers,
        "state_dict": selector.state_dict(),
        "seq_len": seq_len,
        "pred_len": pred_len,
        "best_epoch": selector_best_epoch,
    }, ckpt_path)

    np.savez_compressed(
        pred_path,
        preds=pred,
        trues=trues,
        probs=sel_eval["probs"],
        seq_len=seq_len,
        pred_len=pred_len,
    )

    rows.append({
        "dataset": dataset_name,
        "mode": "selector",
        "model": "CaPMeS_selector",
        "family": "meta_selection_expert_pool",
        "seq_len": seq_len,
        "pred_len": pred_len,
        "batch_size": resolve_batch_size(dataset_name),
        "best_epoch": selector_best_epoch,
        "test_mse": float(sel_mse),
        "test_mae": float(sel_mae),
        "test_rmse": float(sel_rmse),
        "avg_selection_probs_json": json.dumps(sel_eval["avg_probs"].tolist()),
        "prediction_file": str(pred_path.relative_to(ROOT)),
        "checkpoint": str(ckpt_path.relative_to(ROOT)),
    })

    # expert prediction files (optional save for reproducibility)
    for r in expert_runs:
        epred_path = OUT_PREDS / f"{dataset_name}_{r['expert']}_test_predictions.npz"
        np.savez_compressed(
            epred_path,
            preds=r["test_eval"]["preds"],
            trues=r["test_eval"]["trues"],
            seq_len=seq_len,
            pred_len=pred_len,
        )

    # cleanup
    del selector
    for r in expert_runs:
        del r["model"]
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return pd.DataFrame(rows)

In [ ]:
# ============================================
# PILOT RUN
# ============================================

pilot_dfs = []

if RUN_MODE in ["pilot_only", "pilot_then_full"]:
    print("Starting pilot...")
    for ds in PILOT_DATASETS:
        print("\n" + "#" * 120)
        print(f"PILOT DATASET: {ds}")
        print("#" * 120)
        df_ds = run_dataset(ds)
        pilot_dfs.append(df_ds)

    pilot_df = pd.concat(pilot_dfs, ignore_index=True)

    pilot_metrics_csv = OUT_TABLES / "pilot_metrics.csv"
    pilot_df.to_csv(pilot_metrics_csv, index=False)

    print("\nSaved:", pilot_metrics_csv)
    display(pilot_df)

    # summarize
    pivot = pilot_df.pivot(index="dataset", columns="mode", values="test_rmse").reset_index()

    expert_cols = [c for c in pivot.columns if c not in ["dataset", "selector"]]
    pivot["best_internal_expert_rmse"] = pivot[expert_cols].min(axis=1)
    pivot["best_internal_expert_name"] = pivot[expert_cols].idxmin(axis=1)

    pivot["selector_vs_best_internal_gain"] = pivot["best_internal_expert_rmse"] - pivot["selector"]
    pivot["selector_vs_dlinear_gain"] = pivot["dlinear_target"] - pivot["selector"]

    best_internal_wins = int((pivot["selector_vs_best_internal_gain"] > 0).sum())
    dlinear_wins = int((pivot["selector_vs_dlinear_gain"] > 0).sum())

    pilot_summary = pivot.copy()
    pilot_summary["pilot_pass_best_internal"] = best_internal_wins
    pilot_summary["pilot_pass_dlinear"] = dlinear_wins

    pilot_summary_csv = OUT_TABLES / "pilot_summary.csv"
    pilot_summary.to_csv(pilot_summary_csv, index=False)

    print("\nSaved:", pilot_summary_csv)
    display(pilot_summary)

    PILOT_GOOD = (
        (best_internal_wins >= PILOT_THRESH_BEST_INTERNAL)
        and
        (dlinear_wins >= PILOT_THRESH_DLINEAR)
    )

    print("\nPILOT DECISION")
    print(f"selector beats strongest internal expert on {best_internal_wins}/{len(PILOT_DATASETS)} pilot datasets")
    print(f"selector beats dlinear_target on {dlinear_wins}/{len(PILOT_DATASETS)} pilot datasets")
    print("PILOT_GOOD =", PILOT_GOOD)
else:
    PILOT_GOOD = False

In [ ]:
# ============================================
# FULL 9-DATASET RUN
# ============================================

RUN_FULL = (
    (RUN_MODE == "full_only")
    or
    (RUN_MODE == "pilot_then_full" and AUTO_FULL_IF_PILOT_GOOD and PILOT_GOOD)
)

print("RUN_FULL =", RUN_FULL)

if RUN_FULL:
    full_dfs = []
    for ds in ALL_DATASETS:
        print("\n" + "#" * 120)
        print(f"FULL DATASET: {ds}")
        print("#" * 120)
        df_ds = run_dataset(ds)
        full_dfs.append(df_ds)

    full_df = pd.concat(full_dfs, ignore_index=True)

    full_metrics_csv = OUT_TABLES / "full_metrics.csv"
    full_df.to_csv(full_metrics_csv, index=False)

    print("\nSaved:", full_metrics_csv)
    display(full_df)

    if BEST_PATH.exists():
        best_old = pd.read_csv(BEST_PATH)
        selector_rows = full_df[full_df["mode"] == "selector"].copy()

        compare = best_old.rename(columns={
            "model": "old_best_model",
            "family": "old_best_family",
            "test_mse": "old_best_mse",
            "test_mae": "old_best_mae",
            "test_rmse": "old_best_rmse",
        }).merge(
            selector_rows[["dataset", "model", "test_mse", "test_mae", "test_rmse", "avg_selection_probs_json"]].rename(columns={
                "model": "selector_model",
                "test_mse": "selector_mse",
                "test_mae": "selector_mae",
                "test_rmse": "selector_rmse",
            }),
            on="dataset",
            how="left"
        )

        compare["rmse_gain_abs"] = compare["old_best_rmse"] - compare["selector_rmse"]
        compare["rmse_gain_pct"] = 100.0 * compare["rmse_gain_abs"] / compare["old_best_rmse"]

        compare["mae_gain_abs"] = compare["old_best_mae"] - compare["selector_mae"]
        compare["mae_gain_pct"] = 100.0 * compare["mae_gain_abs"] / compare["old_best_mae"]

        compare_csv = OUT_TABLES / "full_vs_current_best.csv"
        compare.to_csv(compare_csv, index=False)

        wins = int((compare["rmse_gain_abs"] > 0).sum())
        losses = int((compare["rmse_gain_abs"] < 0).sum())
        ties = int((compare["rmse_gain_abs"] == 0).sum())

        print("\nSaved:", compare_csv)
        display(compare)

        print("\nFINAL DECISION")
        print(f"CaPMeS selector wins on {wins}/{len(ALL_DATASETS)} datasets by RMSE")
        print(f"Loses on {losses}/{len(ALL_DATASETS)} datasets")
        print(f"Ties on {ties}/{len(ALL_DATASETS)} datasets")
else:
    print("Full run skipped. Stop here if pilot failed.")

## Exact module changes

1. Train **independent experts** instead of one joint architecture:
   - `seasonal_naive`
   - `dlinear_target`
   - `local_conv`
   - `wavelet_expert`

2. Add **selector feature extraction**:
   - trend/variance/slope
   - lag-1 autocorrelation
   - FFT high-frequency ratio
   - Haar multiscale summary

3. Add **prototype-conditioned selector**:
   - fit KMeans on training-window selector features
   - feed selector both features and distances to prototypes

4. Add **calibrated meta-selection**:
   - hard label = best expert on validation window
   - soft target = normalized negative expert errors
   - selective weighting = keep higher-margin validation windows

## Exact pseudocode

1. Train each expert separately on train split.
2. Run all experts on validation split.
3. Build meta-dataset:
   - input = selector features
   - hard target = expert with lowest validation error
   - soft target = softmax(-errors / tau)
   - weight = confidence margin mask
4. Fit prototypes on training windows.
5. Train selector on validation meta-dataset.
6. On test:
   - run all experts
   - selector chooses or softly combines experts
   - compare against strongest internal expert and current unified best

## Forward-pass equations

Experts:
\[
\hat y_k = f_k(x), \quad k \in \{1,\dots,K\}
\]

Selector features:
\[
z = \phi(x)
\]

Prototype distances:
\[
d_j = \|z - c_j\|_2
\]

Selector logits:
\[
\ell = g([z, d_1, \dots, d_M])
\]

Hard selection:
\[
k^* = \arg\max \text{softmax}(\ell)
\]
\[
\hat y = \hat y_{k^*}
\]

Soft selection:
\[
\pi = \text{softmax}(\ell / \tau)
\]
\[
\hat y = \sum_k \pi_k \hat y_k
\]

## Loss equations

Hard target:
\[
y^{hard} = \arg\min_k \, \text{Err}_k
\]

Soft target:
\[
y^{soft}_k = \frac{\exp(-\text{Err}_k / \tau_s)}{\sum_j \exp(-\text{Err}_j / \tau_s)}
\]

Weighted hard CE:
\[
\mathcal{L}_{hard}
= \frac{\sum_i m_i \, \text{CE}(\ell_i, y^{hard}_i)}{\sum_i m_i + \epsilon}
\]

Soft distillation:
\[
\mathcal{L}_{soft}
= \text{KL}(\text{softmax}(\ell), y^{soft})
\]

Entropy regularization:
\[
\mathcal{L}_{ent}
= -\mathbb{E}\left[\sum_k p_k \log p_k\right]
\]

Total:
\[
\mathcal{L}
= \mathcal{L}_{hard}
+ \lambda_{soft} \mathcal{L}_{soft}
+ \lambda_{ent} \mathcal{L}_{ent}
\]

## Exact ablations

Run:
- `seasonal_naive`
- `dlinear_target`
- `local_conv`
- `wavelet_expert`
- `selector`

Then ablate:
- hard vs soft selection
- no prototypes
- no soft targets
- no selective weighting
- fewer/more prototypes

## Cheapest pilot first

Run only:
- ETTh1
- Weather
- Traffic
- ILI

Continue only if:
- selector beats strongest internal expert on at least 3/4
- selector beats dlinear_target on at least 3/4

## Final 9-dataset schedule

Only if pilot passes, run:
ETTh1, ETTh2, ETTm1, ETTm2, Weather, Electricity, Traffic, Exchange, ILI

## Failure signals to monitor

Reject if:
- selector loses to strongest internal expert in pilot
- selector collapses to always selecting one expert everywhere
- wavelet expert is never selected
- full run wins on too few datasets

## Expected metrics table layout

- `dataset`
- `mode`
- `model`
- `family`
- `seq_len`
- `pred_len`
- `batch_size`
- `best_epoch`
- `test_mse`
- `test_mae`
- `test_rmse`
- `avg_selection_probs_json`

## Result threshold that justifies continuing

Continue only if:
- pilot passes, and
- full run beats current unified best on at least 4/9 datasets,
or
- clearly improves hard regimes with interpretable expert selection.